# 01 · Data understanding
Run `python -m src.pipeline` first. This notebook reads the Silver and Gold layers.

**Questions:** how big is each table, how imbalanced is `TARGET`, how much history does each applicant have, and which data issues did the Silver layer fix?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd
GOLD, SILVER, REPORTS = ROOT / "data/gold", ROOT / "data/silver", ROOT / "reports/data_studio/data"
def q(sql: str) -> pd.DataFrame:
    return duckdb.sql(sql.replace("{gold}", GOLD.as_posix()).replace("{silver}", SILVER.as_posix())).df()

In [ ]:
sizes = {p.stem: q(f"SELECT count(*) AS n FROM '{p.as_posix()}'")["n"][0]
         for p in sorted(SILVER.glob("*.parquet"))}
pd.Series(sizes, name="rows").map("{:,}".format)

## Target balance

In [ ]:
q("SELECT TARGET, count(*) AS n, count(*) / sum(count(*)) OVER () AS share FROM '{silver}/application.parquet' GROUP BY 1")

## Coverage of the history tables
Share of applicants with bureau history, previous applications and installment records.

In [ ]:
q("""SELECT avg(HAS_BUREAU_HISTORY) AS with_bureau,
          avg((PREV_N_APPLICATIONS IS NOT NULL)::INT) AS with_previous_app,
          avg((INST_N_PAYMENTS IS NOT NULL)::INT) AS with_installments,
          avg((BB_N_MONTHS IS NOT NULL)::INT) AS with_monthly_bureau_status
   FROM '{gold}/client_features.parquet'""")

## Cleaning applied in Silver
`DAYS_EMPLOYED = 365243` is a sentinel value (mostly pensioners). It is set to NULL and flagged.

In [ ]:
q("""SELECT NAME_INCOME_TYPE, count(*) AS n, avg(FLAG_DAYS_EMPLOYED_ANOM) AS share_sentinel,
          avg(TARGET) AS default_rate
   FROM '{silver}/application.parquet' GROUP BY 1 ORDER BY n DESC""")

## Notes
_Write 3–5 observations here after running the notebook._